In [4]:
import pandas as pd

In [5]:
path = "../data/df_model.csv"

df = pd.read_csv(path, low_memory=False)

df.head()

,encounter_id,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,...,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,severity,total_visits
0,2278392,8222157,Caucasian,Female,[0-10),6,25,1,1,NaN,...,No,No,No,No,No,No,No,0,2,0
1,149190,55629189,Caucasian,Female,[10-20),1,1,7,3,NaN,...,No,No,No,No,No,Ch,Yes,0,30,0
2,64410,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,NaN,...,No,No,No,No,No,No,Yes,0,14,3
3,500364,82442376,Caucasian,Male,[30-40),1,1,7,2,NaN,...,No,No,No,No,No,Ch,Yes,0,16,0
4,16680,42519267,Caucasian,Male,[40-50),1,1,7,1,NaN,...,No,No,No,No,No,Ch,Yes,0,6,0


## Splitting Data

In [6]:
from sklearn.model_selection import train_test_split

X = df.drop("readmitted", axis=1)
y = df["readmitted"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=631, stratify=y)

## Transforming

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

In [8]:
num_attribs = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_attribs = X_train.select_dtypes(include=["object"]).columns.tolist()

num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy="median")),
        ('std_scaler', StandardScaler()),
    ])

full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_attribs),
])

X_train_prepared = full_pipeline.fit_transform(X_train)
X_test_prepared = full_pipeline.transform(X_test)

## Models

In [9]:
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, accuracy_score

In [10]:
def model_results(y_test, y_pred, name):
    print(f"=={name}==\n")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("\nPrecision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("\nF1 Score:", f1_score(y_test, y_pred))
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [11]:
model = GaussianNB()

# have to convert to array for gaussian
X_train_dense = X_train_prepared.toarray()
X_test_dense = X_test_prepared.toarray()

model.fit(X_train_dense, y_train)

y_pred = model.predict(X_test_dense)

model_results(y_test, y_pred, "GaussianNB")

==GaussianNB==

Accuracy: 0.1538272575415152

Precision: 0.11368334022323275
Recall: 0.968736239542052

F1 Score: 0.20348702770198399

Confusion Matrix:
 [[  931 17152]
 [   71  2200]]


In [12]:
model = RandomForestClassifier(n_estimators=100, random_state=631)

model.fit(X_train_prepared, y_train)

y_pred = model.predict(X_test_prepared)

model_results(y_test, y_pred, "Random Forest Classifier")

==Random Forest Classifier==

Accuracy: 0.8884248796305394

Precision: 0.5
Recall: 0.003963011889035667

F1 Score: 0.007863695937090432

Confusion Matrix:
 [[18074     9]
 [ 2262     9]]


In [13]:
model = KNeighborsClassifier(n_neighbors=5)

model.fit(X_train_prepared, y_train)

y_pred = model.predict(X_test_prepared)

model_results(y_test, y_pred, "K-Nearest Neighbors")

==K-Nearest Neighbors==

Accuracy: 0.8778618453375258

Precision: 0.2879684418145957
Recall: 0.06428885953324527

F1 Score: 0.10511159107271419

Confusion Matrix:
 [[17722   361]
 [ 2125   146]]
